# RadiologyAI — Linha de base honesta no Google Colab

**O que este notebook produz:** *um* número de desempenho real, medido, reproduzível,
com intervalo de confiança — e o artefato `metrics.json` que o sustenta.

É o marco da Fase 1 do [ROADMAP](https://github.com/drguilhermecapel/radiologyai/blob/main/ROADMAP.md):
substituir todas as métricas fabricadas do repositório v1 por uma medição verificável.

**Como este notebook falha:** parando. Cada passo é uma chamada Python no kernel; um
erro interrompe *Executar tudo* na célula certa, com o traceback certo. A versão
anterior usava células `!shell`, que engolem erros — a avaliação falhava em silêncio
e o notebook morria três células depois com um `IndexError` sem relação com a causa.

---

## A armadilha que este notebook evita

O caminho óbvio seria rodar `densenet121-res224-all` no NIH ChestX-ray14. **Não faça isso.**
Os pesos `-all` foram treinados em NIH, PadChest, CheXpert, MIMIC-CXR, Google, OpenI e RSNA —
o nome do arquivo publicado é `nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-...`.
Avaliá-los no NIH é **in-distribution**: um número inflado que parece medição e não é.

Este notebook usa **`densenet121-res224-pc`** — treinado só em PadChest (Alicante, Espanha)
— avaliado no **split oficial de teste do NIH** (EUA, 25.596 imagens, disjunto por paciente).
**Validação externa genuína, sem credenciamento, custo zero.** O código detecta vazamento
e recusa chamar de validação externa o que não é.

---

## Antes de começar

1. `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`.
2. Sem GPU funciona, mas a inferência leva ~2 h em vez de ~12 min.
3. O disco do Colab é **efêmero**: o dataset é baixado a cada sessão. Só os artefatos
   (poucos KB) persistem. **Mantenha a aba aberta** — o Colab gratuito desconecta por
   inatividade do navegador, e a sessão leva o download junto.


## 1. Ambiente


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

print('Python', sys.version.split()[0])

if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True)
    print('GPU:', gpu.stdout.strip() or 'nenhuma — vai rodar em CPU, só mais devagar')
else:
    print('GPU: nenhuma — vai rodar em CPU, só mais devagar')

raiz = '/content' if Path('/content').exists() else '/'
disco = subprocess.run(['df', '-h', raiz], capture_output=True, text=True)
print('disco:', disco.stdout.strip().splitlines()[-1])


### Código

Clona o repositório na branch de trabalho. Se a variável de ambiente
`RADIOLOGYAI_REPO_DIR` apontar para um clone existente (por exemplo no Drive), usa-o.

`PYTHONPATH` é definido **para o processo do kernel e para qualquer subprocesso** — é
isso que o notebook anterior não fazia, e o motivo de os comandos `!python -m ...`
falharem com `No module named radiologyai`.


In [ ]:
REPO = 'https://github.com/drguilhermecapel/radiologyai.git'
BRANCH = 'claude/roadmap-interpretacao-radiologica-vh15ek'

REPO_DIR = Path(os.environ.get('RADIOLOGYAI_REPO_DIR', '/content/radiologyai'))

if not (REPO_DIR / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(REPO_DIR)],
                   check=True)
elif 'RADIOLOGYAI_REPO_DIR' not in os.environ:
    # Clone padrão já existe (execução anterior nesta sessão): traz as correções
    # mais recentes da branch. Um clone fornecido por variável de ambiente é do
    # usuário e NÃO é tocado.
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'FETCH_HEAD'], check=True)
    print('clone existente atualizado para a ponta da branch')

os.chdir(REPO_DIR)
SRC = str(REPO_DIR / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC   # herdado por subprocessos

print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
print('repositório:', REPO_DIR)


### Dependências

O Colab traz `torch` e `numpy`; instalamos o resto. O pacote é instalado com
`--ignore-requires-python`: o `pyproject` declara `>=3.11,<3.14` para os ambientes
testados no CI, e o Colab pode estar numa versão mais nova. A instalação é feita por
`subprocess.run(check=True)`: **se o pip falhar, a célula falha** — nada de seguir
adiante com o pacote pela metade.


In [ ]:
def pip(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    print('$', ' '.join(cmd[3:]))
    subprocess.run(cmd, check=True)

pip('pydicom==2.4.4', 'pydantic>=2.7', 'pydantic-settings>=2.3', 'typer>=0.12',
    'torchxrayvision', 'scikit-image', 'pillow', 'PyYAML')
pip('--no-deps', '--ignore-requires-python', '-e', '.')

# Se este kernel já importou uma versão anterior do pacote (re-execução após
# atualizar o clone), descarta os módulos em cache — importlib.reload() só
# recarrega o pacote raiz, não os submódulos.
for nome in [m for m in sys.modules if m == 'radiologyai' or m.startswith('radiologyai.')]:
    del sys.modules[nome]
import radiologyai
print()
print('radiologyai', radiologyai.__version__)
print(radiologyai.selftest())


## 2. Onde salvar os artefatos

O dataset **não** vai para o Drive: 42 GB não cabem na conta gratuita, e não precisam —
os pixels são reproduzíveis a partir da fonte. O que persiste é o `metrics.json` e o
manifest, poucos megabytes que são o registro de reprodutibilidade.

Montar o Drive abre uma janela de autorização. **Para rodar sem supervisão, ponha
`USAR_DRIVE = False`** — os artefatos ficam no disco do Colab e a última seção oferece o
download. O risco: se a sessão cair, o resultado vai junto.


In [ ]:
USAR_DRIVE = True

ARTIFACTS = Path(os.environ.get(
    'RADIOLOGYAI_ARTIFACTS_DIR',
    '/content/artifacts/eval' if Path('/content').exists() else str(REPO_DIR / 'artifacts' / 'eval'),
))

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        ARTIFACTS = Path('/content/drive/MyDrive/radiologyai/artifacts/eval')
    except Exception as erro:
        print(f'Drive não montado ({type(erro).__name__}: {erro}). Usando o disco local.')
        print('O resultado será perdido se a sessão cair — baixe-o ao final.')

ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('artefatos em:', ARTIFACTS)


## 3. Dados — NIH ChestX-ray14

112.120 radiografias frontais de 30.805 pacientes. Livre, sem credenciamento.

### Credencial do Kaggle: sem arquivo `kaggle.json`

A CLI do Kaggle lê `KAGGLE_USERNAME` e `KAGGLE_KEY` do ambiente, e a célula abaixo as
pega do **Colab Secrets** (ícone de chave, barra lateral esquerda). Crie os dois secrets
com *Notebook access* ligado:

| Secret | Valor |
|---|---|
| `KAGGLE_USERNAME` | seu usuário do Kaggle |
| `KAGGLE_KEY` | o campo `key` do `kaggle.json` gerado em kaggle.com/settings → *Create New Token* |

### Rotas

| | Kaggle | NIH oficial |
|---|---|---|
| Credencial | Colab Secrets | **nenhuma** |
| Arquivos | `Data_Entry_2017.csv` + `images_0xx/` | `Data_Entry_2017_v2020.csv` + `images_0xx/` |
| Disco | ~45 GB | ~10 GB com `--test-only` |

Os dois nomes de CSV são aceitos. `ROTA = 'auto'` usa o Kaggle quando há credencial e
cai para o NIH quando não há. Se `RADIOLOGYAI_DATA_ROOT` apontar para um diretório que
já contém os dados, nada é baixado.


In [ ]:
ROTA = 'auto'   # 'auto' | 'kaggle' | 'nih'

DATA = Path(os.environ.get('RADIOLOGYAI_DATA_ROOT', '/content/nih'))
DATA.mkdir(parents=True, exist_ok=True)


def dados_presentes(raiz: Path) -> bool:
    csv_ok = any((raiz / n).exists() for n in ('Data_Entry_2017_v2020.csv', 'Data_Entry_2017.csv'))
    return csv_ok and (raiz / 'test_list.txt').exists() and any(raiz.rglob('*.png'))


def credencial_kaggle() -> bool:
    """Lê KAGGLE_USERNAME/KAGGLE_KEY do Colab Secrets. Nenhum kaggle.json é necessário."""
    try:
        from google.colab import userdata
    except ImportError:
        return False
    for u, k in [('KAGGLE_USERNAME', 'KAGGLE_KEY'), ('kaggle_username', 'kaggle_key')]:
        try:
            usuario, chave = userdata.get(u), userdata.get(k)
        except Exception:
            continue
        if usuario and chave:
            os.environ['KAGGLE_USERNAME'] = usuario.strip()
            os.environ['KAGGLE_KEY'] = chave.strip()
            print(f'Credencial do Kaggle lida do Secrets (usuário: {usuario.strip()})')
            return True
    return False


if dados_presentes(DATA):
    print('Dados já presentes em', DATA, '— nada a baixar.')
else:
    usar_kaggle = ROTA == 'kaggle' or (ROTA == 'auto' and credencial_kaggle())
    if ROTA == 'kaggle' and not credencial_kaggle():
        raise RuntimeError('ROTA=kaggle sem credencial no Secrets. Crie KAGGLE_USERNAME e KAGGLE_KEY, ou use ROTA="nih".')
    if usar_kaggle:
        print('Rota: Kaggle (~45 GB)')
        pip('kaggle')
        subprocess.run(['kaggle', 'datasets', 'download', '-d', 'nih-chest-xrays/data',
                        '-p', str(DATA), '--unzip'], check=True)
    else:
        print('Rota: NIH oficial, sem credencial. (Com KAGGLE_USERNAME/KAGGLE_KEY no Secrets é mais rápido.)')
        subprocess.run([sys.executable, 'scripts/fetch_nih_cxr14.py', '--out', str(DATA), '--test-only'],
                       check=True)


In [ ]:
# Verificação de integridade ANTES de qualquer inferência. Falha aqui é barata;
# descobrir depois de horas que o CSV estava truncado não é.
from radiologyai.data.nih_cxr14 import find_data_entry_csv

csv = find_data_entry_csv(DATA)
n_csv = len(csv.read_text(encoding='utf-8', errors='replace').splitlines()) - 1
n_test = len([x for x in (DATA / 'test_list.txt').read_text().splitlines() if x.strip()])
n_png = sum(1 for _ in DATA.rglob('*.png'))

print(f'{csv.name}: {n_csv} linhas (oficial: 112120)')
print(f'test_list.txt: {n_test} imagens (oficial: 25596)')
print(f'imagens em disco: {n_png}')

if n_test == 25596 and n_png < n_test:
    raise RuntimeError(f'apenas {n_png} imagens em disco para {n_test} do split de teste: download incompleto')


## 4. Rodar a avaliação

Uma chamada de função. Constrói o manifest a partir do split oficial de teste, verifica
vazamento contra os dados de treino do modelo, verifica o sha256 dos pesos, roda a
inferência e grava `artifacts/eval/<run_id>/metrics.json` com `git_sha`, `weights_sha256`,
`manifest_sha256`, `seed` e versões — o necessário para demonstrar reprodutibilidade
(IEC 62304 §5.7).

~12 min em T4, ~2 h em CPU. Qualquer erro **para aqui**, com o traceback.


In [ ]:
import torch
from radiologyai.evaluation.baseline import run_baseline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('dispositivo:', DEVICE)

result = run_baseline(
    DATA,
    ARTIFACTS,
    card_id='xrv-densenet121-pc',
    manifest_path=REPO_DIR / 'datasets' / 'manifests' / 'nih_cxr14_test.csv',
    device=DEVICE,
    batch_size=64 if DEVICE == 'cuda' else 16,
    n_bootstrap=2000,
    seed=20260101,
)
run_dir = result.run_dir


## 5. Ler o resultado

**Expectativa: AUROC macro entre 0,72 e 0,82.** Cardiomegalia, derrame e enfisema altos
(0,85–0,90); pneumonia, infiltrado e nódulo baixos (0,65–0,73).

O README do v1 alegava 0,94. **Publicar 0,78 com intervalo de confiança é o objetivo
inteiro deste marco.** Se sair muito acima do esperado, desconfie antes de comemorar.


In [ ]:
from radiologyai.evaluation.baseline import summarize

summarize(result)
m = result.metrics


In [ ]:
# Subgrupos — requisito de equidade. Uma lacuna grande é achado a reportar, não a
# esconder. A diferença PA vs AP costuma ser visível e é ela própria um achado.
for group, values in m['subgroups'].items():
    print(f'\n{group}:')
    for k, v in values.items():
        print(f"  {k:<12} n={v['n']:>6}  AUROC macro={v['macro_auroc']}")

print('\nLIMITAÇÕES DECLARADAS:')
for l in m['limitations']:
    print(' *', l)


## 6. Levar o resultado de volta ao repositório

O artefato é o que autoriza qualquer alegação. Sem ele, `scripts/check_honesty.py`
quebra o build de qualquer documento que cite um número.


In [ ]:
import shutil

dest = REPO_DIR / 'artifacts' / 'eval' / run_dir.name
if dest.resolve() != run_dir.resolve():
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)

# O guardião precisa aprovar o repositório com o artefato dentro.
subprocess.run([sys.executable, 'scripts/check_honesty.py'], check=True)
print()
print(subprocess.run(['git', 'status', '--short', 'artifacts/', 'datasets/'],
                     capture_output=True, text=True).stdout or '(nada novo no git — artefatos podem estar no .gitignore, exceto metrics.json)')


In [ ]:
# Baixe o artefato para o seu computador. Faça isto SEMPRE que não usar o Drive.
zip_path = shutil.make_archive(str(run_dir.parent / 'baseline'), 'zip', run_dir)
print('zip:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as erro:
    print(f'Download automático indisponível ({type(erro).__name__}). Pegue o zip pelo painel de arquivos, à esquerda.')


---

## O que este resultado é — e o que não é

**É:** uma medição retrospectiva de desempenho de algoritmo isolado, num conjunto de
teste externo ao treino do modelo, com intervalo de confiança e análise de subgrupos.

**Não é:** validação clínica. Não é evidência de utilidade clínica. Não é medição do
modelo *do produto* — este é um modelo de terceiros usado como referência. Os escores
não são calibrados e não representam probabilidade de doença.

Os rótulos do NIH são **minerados por NLP dos laudos**, não adjudicados por radiologista.
A validação contra rótulos adjudicados vem na Fase 3, com o **VinDr-CXR** — que exige
credenciamento PhysioNet (curso CITI, ~6 h; aprovação em 2–6 semanas). **Comece agora:**
este notebook foi desenhado para não depender de nada credenciado, justamente para que
o credenciamento corra em paralelo e nunca fique no caminho crítico.
